# Aula 6 — Mini-pipeline: juntando tudo

> **Objetivo:** construir um pipeline Extract → Transform → Load completo usando tudo que aprendemos.

---

## O que vamos construir

Um script Python que:

1. **Extrai** dados de um arquivo CSV (simulando uma landing zone)
2. **Transforma** — limpa, valida e enriquece cada registro
3. **Carrega** o resultado em JSON (simulando a camada Bronze)
4. Gera um **relatório de execução**

> 💡 *Na vida real:* esse script seria uma task em um DAG do Airflow. O JSON de saída iria para o S3 e dispararia a próxima etapa do pipeline. A lógica é exatamente a mesma.

## Revisão rápida — os 5 blocos que usaremos

| Aula | Bloco | Onde aparece no pipeline |
|---|---|---|
| 1 | Tipos e operações | Em toda a transformação |
| 2 | Listas e dicionários | Estrutura dos dados |
| 3 | Validação e try/except | Etapa de validação |
| 4 | Funções | Cada etapa é uma função |
| 5 | Arquivos CSV e JSON | Extração e carga |

## Passo 1 — Preparar o ambiente e criar os dados de entrada

In [3]:
import csv
import json
import os
from datetime import datetime

# Criar estrutura de pastas (simula landing zone e bronze)
os.makedirs('landing', exist_ok=True)
os.makedirs('bronze', exist_ok=True)
os.makedirs('logs', exist_ok=True)

# Dados de entrada (simulando arquivo que chegou da fonte)
dados_entrada = """id,nome,email,cidade,valor_compra,data_compra
1001,Ana Lima,ana@email.com,São Paulo,1850.00,20/01/2024
1002,Bruno Costa,,Rio de Janeiro,320.50,20/01/2024
1003,Carla Dias,carla@email.com,Belo Horizonte,4200.00,20/01/2024
1004,Diego Nunes,diego@email.com,Porto Alegre,-50.00,20/01/2024
1005,Eva Melo,eva@email.com,Recife,,20/01/2024
1006,Felipe Rosa,felipe@email.com,Fortaleza,890.00,20/01/2024
1007,,joao@email.com,Manaus,1200.00,20/01/2024
1008,Gisele Tuno,gisele@email.com,Salvador,670.00,data_invalida
"""

with open('landing/pedidos_20240120.csv', 'w', encoding='utf-8') as f:
    f.write(dados_entrada)

print('✓ Arquivo de entrada criado: landing/pedidos_20240120.csv')
print(f'  Linhas: {len(dados_entrada.strip().splitlines()) - 1} registros + cabeçalho')

✓ Arquivo de entrada criado: landing/pedidos_20240120.csv
  Linhas: 8 registros + cabeçalho


**📝 O que essa célula faz:**

Prepara o ambiente: cria as pastas que vão simular as camadas do Data Lake, e escreve um arquivo CSV de entrada com dados **propositalmente sujos** (campos vazios, valores negativos, nomes ausentes, data inválida).

```python
os.makedirs('landing', exist_ok=True)
os.makedirs('bronze', exist_ok=True)
os.makedirs('logs', exist_ok=True)
```

**🗂️ Por que três pastas diferentes?**

Elas simulam três zonas reais de um pipeline de dados:
- `landing/` → onde o arquivo bruto chega, sem nenhum tratamento (equivalente à "landing zone" de um Data Lake real)
- `bronze/` → onde o dado processado e limpo é salvo (equivalente à camada Bronze)
- `logs/` → onde fica o histórico de execuções do pipeline

**🧪 Por que os dados de entrada têm erros propositais?** Olhando o CSV: o registro `1002` não tem email, o `1004` tem valor **negativo**, o `1005` não tem valor, o `1007` não tem nome, e o `1008` tem uma data inválida (`data_invalida`). Isso simula a realidade — dados do mundo real quase nunca chegam perfeitos, e o pipeline que construiremos precisa lidar com cada um desses problemas.

## Passo 2 — Funções de transformação (camada de lógica de negócio)

In [4]:
# ─── Funções de limpeza ──────────────────────────────────────────

def limpar_texto(texto):
    """Remove espaços extras e padroniza para minúsculas."""
    if not texto or texto.strip() == '':
        return None
    return texto.strip()

def converter_valor(texto):
    """Converte texto de valor monetário para float."""
    if not texto or texto.strip() == '':
        return None
    try:
        return float(texto.replace(',', '.'))
    except ValueError:
        return None

def converter_data(texto):
    """Converte 'dd/mm/aaaa' para 'aaaa-mm-dd'."""
    if not texto:
        return None
    try:
        partes = texto.split('/')
        if len(partes) != 3:
            return None
        dia, mes, ano = partes
        return f'{ano}-{mes.zfill(2)}-{dia.zfill(2)}'
    except Exception:
        return None


# ─── Função de validação ─────────────────────────────────────────

def validar_registro(rec):
    """Retorna (valido: bool, motivo: str)."""
    if not rec.get('nome'):
        return False, 'nome ausente'
    if not rec.get('email'):
        return False, 'email ausente'
    if rec.get('valor_compra') is None:
        return False, 'valor ausente'
    if rec['valor_compra'] < 0:
        return False, 'valor negativo'
    if rec.get('data_compra') is None:
        return False, 'data inválida'
    return True, 'ok'


# ─── Função de enriquecimento ─────────────────────────────────────

def classificar_cliente(valor):
    """Classifica o cliente com base no valor da compra."""
    if valor >= 2000:
        return 'vip'
    elif valor >= 500:
        return 'regular'
    else:
        return 'ocasional'


print('✓ Funções de transformação carregadas')

✓ Funções de transformação carregadas


**📝 O que essa célula faz:**

Define **quatro funções pequenas e independentes**, cada uma responsável por um tipo de transformação ou checagem específica — reaproveitando os mesmos padrões já vistos nas Aulas 3 e 4.

**🧼 As três funções de limpeza** (`limpar_texto`, `converter_valor`, `converter_data`) seguem o mesmo padrão: recebem um texto bruto, tentam converter/limpar dentro de um `try`, e devolvem `None` se algo der errado.

**✅ `validar_registro(rec)`** — note que ela já espera receber um dicionário **com os campos já convertidos** (não mais texto puro), porque ela testa `rec['valor_compra'] < 0`, uma comparação numérica que só funciona se o valor já foi convertido para número. Ela checa, em sequência: nome ausente, email ausente, valor ausente, valor negativo, data inválida — devolvendo `(False, motivo)` no primeiro problema encontrado, ou `(True, 'ok')` se passar por tudo.

**🏷️ `classificar_cliente(valor)`** — usa `if/elif/else` simples para colocar o cliente em uma faixa (`'vip'`, `'regular'`, `'ocasional'`) dependendo do valor da compra.

**💡 Por que separar em funções pequenas em vez de uma função gigante?** Porque cada uma faz **uma única coisa** e pode ser testada e reaproveitada separadamente. Isso é o princípio de responsabilidade única, citado na Aula 4 — e é exatamente como pipelines profissionais são estruturados.

## Passo 3 — Extract (Extração)

In [5]:
def extrair(caminho_arquivo):
    """Lê o CSV de entrada e retorna lista de dicionários."""
    registros = []
    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        leitor = csv.DictReader(f)
        for linha in leitor:
            registros.append(dict(linha))
    return registros

# Executar extração
registros_brutos = extrair('landing/pedidos_20240120.csv')
print(f'✓ Extração concluída: {len(registros_brutos)} registros lidos')
print('  Exemplo do primeiro registro (bruto):')
print(' ', registros_brutos[0])

✓ Extração concluída: 8 registros lidos
  Exemplo do primeiro registro (bruto):
  {'id': '1001', 'nome': 'Ana Lima', 'email': 'ana@email.com', 'cidade': 'São Paulo', 'valor_compra': '1850.00', 'data_compra': '20/01/2024'}


**📝 O que essa célula faz:**

Define a etapa de **Extract** (extração) do pipeline: lê o arquivo CSV de entrada e devolve uma lista de dicionários.

```python
def extrair(caminho_arquivo):
    registros = []
    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        leitor = csv.DictReader(f)
        for linha in leitor:
            registros.append(dict(linha))
    return registros
```

**🔁 Isso é exatamente o mesmo padrão da Aula 5** (leitura de CSV com `csv.DictReader`), só que agora **encapsulado dentro de uma função** — o que permite chamá-la de forma simples (`extrair('landing/pedidos_20240120.csv')`) em qualquer ponto do pipeline, sem repetir a lógica de abertura de arquivo.

**📦 `dict(linha)`** → o `csv.DictReader` já devolve algo parecido com um dicionário, mas tecnicamente é um tipo especial chamado `OrderedDict`. Envolvê-lo com `dict(...)` converte explicitamente para um dicionário Python comum, evitando comportamentos inesperados em etapas posteriores do pipeline.

## Passo 4 — Transform (Transformação)

In [6]:
def transformar(registros_brutos):
    """
    Aplica limpeza, validação e enriquecimento.
    Retorna: (registros_validos, registros_rejeitados)
    """
    validos = []
    rejeitados = []
    
    for bruto in registros_brutos:
        # 1. Limpar e tipar cada campo
        rec = {
            'id': bruto.get('id'),
            'nome': limpar_texto(bruto.get('nome')),
            'email': limpar_texto(bruto.get('email')),
            'cidade': limpar_texto(bruto.get('cidade')),
            'valor_compra': converter_valor(bruto.get('valor_compra')),
            'data_compra': converter_data(bruto.get('data_compra')),
        }
        
        # 2. Validar
        ok, motivo = validar_registro(rec)
        
        if ok:
            # 3. Enriquecer apenas os válidos
            rec['classificacao'] = classificar_cliente(rec['valor_compra'])
            rec['processado_em'] = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
            validos.append(rec)
        else:
            rejeitados.append({
                'id': bruto.get('id'),
                'motivo_rejeicao': motivo,
                'registro_original': bruto
            })
    
    return validos, rejeitados

# Executar transformação
validos, rejeitados = transformar(registros_brutos)
print(f'✓ Transformação concluída:')
print(f'  Válidos:   {len(validos)}')
print(f'  Rejeitados: {len(rejeitados)}')
print()
print('Registros válidos:')
for v in validos:
    print(f"  [{v['classificacao'].upper():9}] {v['nome']:<15} | R$ {v['valor_compra']:>8.2f}")

✓ Transformação concluída:
  Válidos:   3
  Rejeitados: 5

Registros válidos:
  [REGULAR  ] Ana Lima        | R$  1850.00
  [VIP      ] Carla Dias      | R$  4200.00
  [REGULAR  ] Felipe Rosa     | R$   890.00


**📝 O que essa célula faz:**

Define a etapa de **Transform** (transformação) do pipeline: aplica limpeza, validação e enriquecimento em cada registro, separando os válidos dos rejeitados.

```python
def transformar(registros_brutos):
    validos = []
    rejeitados = []
    
    for bruto in registros_brutos:
        rec = {
            'id': bruto.get('id'),
            'nome': limpar_texto(bruto.get('nome')),
            ...
        }
        ok, motivo = validar_registro(rec)
        
        if ok:
            rec['classificacao'] = classificar_cliente(rec['valor_compra'])
            rec['processado_em'] = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
            validos.append(rec)
        else:
            rejeitados.append({...})
    
    return validos, rejeitados
```

**🔄 As três etapas dentro do laço `for`, em ordem:**

1. **Limpar e tipar:** cada campo bruto (texto) passa pela função de limpeza correspondente, virando um valor já no tipo correto (texto limpo, número, data formatada)
2. **Validar:** o registro já limpo é checado pela `validar_registro()` da Aula 3 — se algum campo obrigatório está ausente ou inválido
3. **Enriquecer (só se válido):** adiciona dois campos novos que não existiam no dado original: `classificacao` (calculada pela função da célula anterior) e `processado_em` (timestamp de quando o pipeline rodou)

**📅 `datetime.now().strftime('%Y-%m-%dT%H:%M:%S')`** → pega a data e hora atual do computador e a formata como texto no padrão `2024-01-20T14:30:00` (formato ISO 8601, o mesmo usado em APIs e bancos de dados).

**↩️ `return validos, rejeitados`** → devolve **duas listas** de uma vez (igual ao retorno duplo de tupla que vimos na função `validar_registro` da Aula 3).

## Passo 5 — Load (Carga)

In [ ]:
def carregar(validos, rejeitados):
    """Salva os dados processados na camada bronze."""
    # Dados válidos → bronze
    caminho_bronze = 'bronze/pedidos_20240120.json'
    with open(caminho_bronze, 'w', encoding='utf-8') as f:
        json.dump(validos, f, ensure_ascii=False, indent=2)
    
    # Rejeitados → quarentena para análise
    caminho_quarentena = 'bronze/pedidos_20240120_rejeitados.json'
    with open(caminho_quarentena, 'w', encoding='utf-8') as f:
        json.dump(rejeitados, f, ensure_ascii=False, indent=2)
    
    return caminho_bronze, caminho_quarentena

caminho_saida, caminho_rejeitos = carregar(validos, rejeitados)
print(f'✓ Carga concluída:')
print(f'  Bronze:     {caminho_saida}')
print(f'  Quarentena: {caminho_rejeitos}')

**📝 O que essa célula faz:**

Define a etapa de **Load** (carga) do pipeline: salva os registros válidos e os rejeitados em arquivos JSON separados.

```python
def carregar(validos, rejeitados):
    caminho_bronze = 'bronze/pedidos_20240120.json'
    with open(caminho_bronze, 'w', encoding='utf-8') as f:
        json.dump(validos, f, ensure_ascii=False, indent=2)
    
    caminho_quarentena = 'bronze/pedidos_20240120_rejeitados.json'
    with open(caminho_quarentena, 'w', encoding='utf-8') as f:
        json.dump(rejeitados, f, ensure_ascii=False, indent=2)
    
    return caminho_bronze, caminho_quarentena
```

**🗑️ Por que salvar os rejeitados em um arquivo separado, em vez de simplesmente descartá-los?**

Esse é o conceito de **quarentena** (também chamado de "dead letter" em sistemas de mensageria): em vez de simplesmente jogar fora os dados problemáticos, eles são guardados separadamente para que alguém possa investigar depois **por que** falharam, sem travar o restante do pipeline. É assim que pipelines de produção tratam dados ruins — nunca descartando silenciosamente.

## Passo 6 — Relatório de execução

In [ ]:
def gerar_relatorio(registros_brutos, validos, rejeitados):
    """Gera e salva o relatório de execução do pipeline."""
    total = len(registros_brutos)
    
    # Estatísticas
    contagem_por_classificacao = {}
    for rec in validos:
        classe = rec['classificacao']
        contagem_por_classificacao[classe] = contagem_por_classificacao.get(classe, 0) + 1
    
    relatorio = {
        'pipeline': 'ingestao_pedidos',
        'data_arquivo': '2024-01-20',
        'executado_em': datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        'totais': {
            'lidos': total,
            'validos': len(validos),
            'rejeitados': len(rejeitados),
            'taxa_rejeicao': f"{len(rejeitados)/total:.1%}" if total > 0 else '0%'
        },
        'classificacao_clientes': contagem_por_classificacao,
        'motivos_rejeicao': [r['motivo_rejeicao'] for r in rejeitados]
    }
    
    caminho = 'logs/relatorio_20240120.json'
    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(relatorio, f, ensure_ascii=False, indent=2)
    
    return relatorio

relatorio = gerar_relatorio(registros_brutos, validos, rejeitados)

print('=' * 50)
print('RELATÓRIO DE EXECUÇÃO')
print('=' * 50)
print(f"Pipeline:        {relatorio['pipeline']}")
print(f"Registros lidos: {relatorio['totais']['lidos']}")
print(f"Válidos:         {relatorio['totais']['validos']}")
print(f"Rejeitados:      {relatorio['totais']['rejeitados']}")
print(f"Taxa de rejeição:{relatorio['totais']['taxa_rejeicao']}")
print()
print('Clientes por classificação:')
for classe, qtd in relatorio['classificacao_clientes'].items():
    print(f'  {classe}: {qtd}')
print()
print('Motivos de rejeição:')
for motivo in relatorio['motivos_rejeicao']:
    print(f'  • {motivo}')

**📝 O que essa célula faz:**

Gera um relatório com estatísticas da execução do pipeline e o salva como JSON.

```python
def gerar_relatorio(registros_brutos, validos, rejeitados):
    contagem_por_classificacao = {}
    for rec in validos:
        classe = rec['classificacao']
        contagem_por_classificacao[classe] = contagem_por_classificacao.get(classe, 0) + 1
    ...
```

**🔢 O truque `dicionario.get(chave, 0) + 1` para contar ocorrências:**

Essa é uma técnica clássica para contar quantas vezes cada valor aparece, sem precisar checar antes "essa chave já existe?". Veja o passo a passo:

- Primeira vez que aparece `'vip'`: `contagem_por_classificacao.get('vip', 0)` → a chave não existe ainda, então devolve o padrão `0`. Soma `+1` → fica `1`. Esse `1` é guardado na chave `'vip'`.
- Segunda vez que aparece `'vip'`: `contagem_por_classificacao.get('vip', 0)` → agora a chave **já existe** e vale `1`. Soma `+1` → fica `2`.

Isso evita ter que escrever um `if 'vip' in contagem_por_classificacao: ... else: ...` mais verboso para o mesmo resultado.

**📊 O dicionário `relatorio` final** organiza tudo: totais (lidos, válidos, rejeitados, taxa de rejeição), a contagem por classificação, e a lista de motivos de rejeição — formando um resumo completo da execução, pronto para ser consultado depois ou exibido em um dashboard.

**`f"{len(rejeitados)/total:.1%}"`** → mesma formatação de porcentagem vista na Aula 4, aqui usada para calcular a taxa de rejeição.

## Passo 7 — O pipeline completo em uma função

Agora que testamos cada parte, vamos unir tudo em uma única função `executar_pipeline`:

In [ ]:
def executar_pipeline(arquivo_entrada):
    """
    Pipeline completo: Extract → Transform → Load.
    
    Parâmetros:
        arquivo_entrada (str): caminho do CSV na landing zone
    
    Retorna:
        dict: relatório de execução
    """
    print(f'[INÍCIO] Pipeline iniciado: {arquivo_entrada}')
    
    # E: Extract
    brutos = extrair(arquivo_entrada)
    print(f'[EXTRACT] {len(brutos)} registros extraídos')
    
    # T: Transform
    validos, rejeitados = transformar(brutos)
    print(f'[TRANSFORM] {len(validos)} válidos | {len(rejeitados)} rejeitados')
    
    # L: Load
    carregar(validos, rejeitados)
    print('[LOAD] Dados salvos na camada bronze')
    
    # Relatório
    relatorio = gerar_relatorio(brutos, validos, rejeitados)
    print(f"[FIM] Taxa de sucesso: {relatorio['totais']['validos']}/{relatorio['totais']['lidos']}")
    
    return relatorio


# Executar!
resultado = executar_pipeline('landing/pedidos_20240120.csv')

**📝 O que essa célula faz:**

Junta as quatro funções anteriores (`extrair`, `transformar`, `carregar`, `gerar_relatorio`) em **uma única função orquestradora**, que representa o pipeline completo do início ao fim.

```python
def executar_pipeline(arquivo_entrada):
    brutos = extrair(arquivo_entrada)
    validos, rejeitados = transformar(brutos)
    carregar(validos, rejeitados)
    relatorio = gerar_relatorio(brutos, validos, rejeitados)
    return relatorio
```

**🎯 Essa é a ideia central da aula:** cada uma das quatro etapas já foi testada **separadamente** nas células anteriores. Agora, a função `executar_pipeline` apenas **chama** cada uma na ordem correta, passando o resultado de uma como entrada da próxima — isso é literalmente o que significa um "pipeline": uma sequência de etapas conectadas, onde a saída de uma alimenta a entrada da seguinte.

**🏭 Em produção, essa função `executar_pipeline` seria exatamente uma "task" do Airflow** — uma DAG (fluxo de tarefas) simplesmente chamaria essa função em um horário agendado, e cada uma das quatro etapas internas poderia até ser dividida em tasks separadas, dependendo da complexidade do pipeline real.

**🖨️ Os `print()` espalhados pela função** servem como um log simples de acompanhamento — mostrando em tempo real em qual etapa o pipeline está, o que ajuda bastante na hora de depurar problemas.

## Onde este código se encaixa na arquitetura que você conhece

```
     LANDING ZONE                BRONZE                    SILVER
  pedidos.csv (raw)   →   pedidos.json (limpo)   →   (próxima etapa)
        ↑                        ↑
    [EXTRACT]             [TRANSFORM + LOAD]
        └──────────── este pipeline ──────────────┘
```

Na prática:
- Este script seria uma **Python Operator no Airflow** (ou uma task no Prefect)
- A pasta `landing/` seria um bucket S3 `s3://lake/landing/`
- A pasta `bronze/` seria `s3://lake/bronze/`
- O `gerar_relatorio` poderia escrever no CloudWatch, Datadog, ou uma tabela de controle

A **lógica** é a mesma. O que muda é o driver de I/O (boto3 para S3, psycopg2 para PostgreSQL, etc.).

---

## 🏋️ Desafio final — Estender o pipeline

Escolha uma (ou mais) das extensões abaixo:

**Nível 1:** Adicionar uma função `gerar_resumo_csv` que salva um CSV simples com id, nome e classificação de todos os clientes válidos.

**Nível 2:** Modificar a função `validar_registro` para também checar se o email tem `@` (validação básica de formato).

**Nível 3:** Adicionar ao relatório o valor total e o valor médio das compras dos registros válidos.

In [ ]:
# Seu código aqui


---

## O que estudar a seguir

| Tópico | Para que serve |
|---|---|
| **Pandas** | Manipulação eficiente de grandes volumes de dados tabulares |
| **PySpark** | Processamento distribuído em clusters |
| **SQLAlchemy** | Conectar Python a bancos de dados SQL |
| **boto3** | SDK Python para AWS (S3, Glue, Lambda) |
| **Airflow** | Orquestrar pipelines como DAGs |
| **pytest** | Testar suas funções de transformação |

---

## Resumo do curso

| Aula | Conceito central | Onde usar |
|---|---|---|
| 1 | Tipos e operações | Todo o código |
| 2 | Listas e dicionários | Estrutura dos dados |
| 3 | Condições e try/except | Validação |
| 4 | Funções | Transformações reutilizáveis |
| 5 | Arquivos CSV e JSON | Extração e carga |
| 6 | Pipeline ETL completo | Produção |

**Parabéns por concluir o curso! 🐍**